# Week 3: Embeddings

This notebook takes the tokenization and inference processes from last time and abstracts them away to pull out the step before it: embeddings.

We will exclusively tokenize on words from now on rather than characters, which allows us to capture semantic word-level meaning in the embeddings.

---

**What are embeddings?**

First recall the logit vector of next-token probabilities produced by the model.

$$
\underbrace{
\left[\text{I},\ \text{ran},\ \text{up},\ \text{the}\right]
}_{\text{Input sequence}}
\quad
\xrightarrow{\quad \text{Transformer} \quad}
\quad
\underbrace{
\left\{
\begin{aligned}
\text{stairs} &: \phantom{-}5.3\\
\text{hill} &: \phantom{-}5.1\\
\text{street} &: \phantom{-}4.5\\
&\ \vdots\\
\text{dog} &: -4.9\\
\text{and} &: -5.2
\end{aligned}
\right\}
}_{\text{Next-token logits}}
$$


But where do these logits actually come from?

Recall that **every position in the sequence predicts its own next token**. The transformer starts with an embedding for each input token, then repeatedly refines these embeddings through attention and feed-forward layers. The result is a set of **contextual embeddings**: vectors that represent each token in the context of those preceding it.

$$
\underbrace{
\left[\text{I},\ \text{ran},\ \text{up},\ \text{the}\right]
}_{\text{Input tokens}}
\xrightarrow{\quad \text{Embedding lookup} \quad}
\underbrace{
\left[\mathbf e_1,\ \mathbf e_2,\ \mathbf e_3,\ \mathbf e_4\right]
}_{\text{Input embeddings}}
\xrightarrow{\quad \text{Transformer blocks} \quad}
\underbrace{
\left[\mathbf h_1,\ \mathbf h_2,\ \mathbf h_3,\ \mathbf h_4\right]
}_{\text{Contextual embeddings}}
$$

Each contextual embedding can be projected into vocabulary space to produce next-token logits. Last week, we used only the final position to predict what comes after "the":

$$
\underbrace{
\mathbf h_4 =
\begin{bmatrix}
0.21\\
-0.84\\
0.37\\
\vdots\\
0.62
\end{bmatrix}
}_{\substack{\text{Contextual embedding}\\ \mathbb R^{d_{\text{model}}}}}
\quad
\xrightarrow{\quad \mathbf W_{\text{out}} \quad}
\quad
\underbrace{
\left\{
\begin{aligned}
\text{stairs} &: \phantom{-}5.3\\
\text{hill} &: \phantom{-}5.1\\
\text{street} &: \phantom{-}4.5\\
&\ \vdots\\
\text{dog} &: -4.9\\
\text{and} &: -5.2
\end{aligned}
\right\}
}_{\substack{\text{Next-token logits}\\ \mathbb R^{|V|}}}
$$

Notice that the embedding has only $d_{\text{model}}$ dimensions, while the logit vector has one dimension for **every token in the vocabulary** ($|V|$).

So how can a relatively small, dense vector have enough information to manipulate and predict an output space with tens of thousands of possible tokens?  
**That's what we'll look at today.**

In [ ]:
import tokens
import transformer_util as tu
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
encode, decode, VOCAB_SIZE, tokens = tokens.tokenize_file("wikipedia_corpus.txt")

In [ ]:
def get_batch_generator(tokens, batch_size, sequence_length, rng):
    def get_batch():
        window_length = sequence_length + 1
        max_start = len(tokens) - window_length

        if max_start < 0:
            raise ValueError(
                "Token sequence is shorter than sequence_length + 1"
            )

        starts = np.arange(0, max_start + 1, sequence_length)
        rng.shuffle(starts)

        offsets = np.arange(window_length)
        for i in range(0, len(starts) - batch_size + 1, batch_size):
            batch_starts = starts[i:i + batch_size]
            windows = tokens[batch_starts[:, None] + offsets]
            yield windows[:, :-1], windows[:, 1:]

    return get_batch

params = tu.initialize_parameters(
    vocab_size=VOCAB_SIZE,
    d_model=64,
    d_ff=256,
    n_layers=3,
    n_heads=4,
    seed=0,
)

# NOTE: saving this is necessary if training is resumed
adam_state = {
    "momentum": tu.zeros_like_tree(params),
    "squared_grad": tu.zeros_like_tree(params),
    "step": 0,
}

BATCH_SIZE = 8
SEQ_LENGTH = 64
rng = np.random.default_rng(0)
get_batch = get_batch_generator(
    tokens,
    batch_size=BATCH_SIZE,
    sequence_length=SEQ_LENGTH,
    rng=rng,
)

losses = [] # Define losses here so multiple training sessions can append to the same list

In [ ]:
LOAD_MODEL = True

if LOAD_MODEL:
    params, adam_state, encode, decode = tu.load_model("model.dill")
else:
    # NOTE: you can run this cell multiple times to continue training the model
    number_of_steps = 15000

    for step in range(number_of_steps):
        inputs, targets = next(get_batch())

        logits, cache = tu.transformer(inputs, params)

        loss = tu.softmax_cross_entropy(logits, targets)

        grad_logits = tu.softmax_cross_entropy_backward(
            logits,
            targets,
        )

        grads = tu.transformer_backward(
            params,
            cache,
            grad_logits,
        )

        tu.adam_step(
            params,
            grads,
            adam_state,
            learning_rate=3e-4,
        )

        if step % 100 == 0:
            epochs = step/(len(tokens)//BATCH_SIZE)
            print(f"{step=:0{len(str(number_of_steps-1))}d}, {loss=:.4f}, {epochs=:.4f}")

        losses.append(loss)

    plt.plot(losses)
    plt.plot(np.convolve(losses, np.ones(100)/100, mode='valid'))
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.show()

    tu.save_model(params, adam_state, encode, decode, f"model.dill")

In [ ]:
generated_text = tu.generate(
    tokenizer=(encode, decode, VOCAB_SIZE),
    params=params,
    prompt="The transformer is",
    number_of_tokens=100,
    context_length=SEQ_LENGTH,
    temperature=1.0,
    seed=100,
    SHOW_ATTENTION=True
)

# Embeddings and Inside the Transformer: The Residual Stream

Now let's look inside our trained model and trace how embeddings evolve during inference.

Each token ID is mapped to a learned vector through the **embedding matrix**. These vectors enter the **residual stream**, which carries information through the entire transformer.

Rather than replacing the embeddings, each attention and feed-forward (FFN) layer computes an update that gets **added back into the stream**. Normalization keeps the representations stable as they evolve.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1280 1165" width="100%" style="max-width:1280px" font-family="system-ui, -apple-system, Segoe UI, Helvetica, Arial, sans-serif" role="img" aria-label="Diagram: token embeddings flowing through transformer blocks">
<text x="50" y="32" font-size="20" text-anchor="start" font-weight="600" fill="currentColor">How token embeddings evolve through a transformer</text>
<line x1="130" y1="184" x2="130" y2="885" stroke="#3B82F6" stroke-width="3"/>
<line x1="130" y1="885" x2="130" y2="940" stroke="#3B82F6" stroke-width="3" stroke-dasharray="3 6"/>
<line x1="130" y1="940" x2="130" y2="945" stroke="#3B82F6" stroke-width="3"/>
<polygon points="130.0,950.0 126.5,943.0 133.5,943.0" fill="#3B82F6"/>
<line x1="270" y1="184" x2="270" y2="885" stroke="#3B82F6" stroke-width="3"/>
<line x1="270" y1="885" x2="270" y2="940" stroke="#3B82F6" stroke-width="3" stroke-dasharray="3 6"/>
<line x1="270" y1="940" x2="270" y2="945" stroke="#3B82F6" stroke-width="3"/>
<polygon points="270.0,950.0 266.5,943.0 273.5,943.0" fill="#3B82F6"/>
<line x1="410" y1="184" x2="410" y2="885" stroke="#3B82F6" stroke-width="3"/>
<line x1="410" y1="885" x2="410" y2="940" stroke="#3B82F6" stroke-width="3" stroke-dasharray="3 6"/>
<line x1="410" y1="940" x2="410" y2="945" stroke="#3B82F6" stroke-width="3"/>
<polygon points="410.0,950.0 406.5,943.0 413.5,943.0" fill="#3B82F6"/>
<line x1="760" y1="184" x2="760" y2="340" stroke="#3B82F6" stroke-width="3"/>
<polygon points="760.0,345.0 756.5,338.0 763.5,338.0" fill="#3B82F6"/>
<line x1="760" y1="375" x2="760" y2="470" stroke="#3B82F6" stroke-width="3"/>
<polygon points="760.0,475.0 756.5,468.0 763.5,468.0" fill="#3B82F6"/>
<line x1="760" y1="505" x2="760" y2="690" stroke="#3B82F6" stroke-width="3"/>
<polygon points="760.0,695.0 756.5,688.0 763.5,688.0" fill="#3B82F6"/>
<line x1="760" y1="725" x2="760" y2="820" stroke="#3B82F6" stroke-width="3"/>
<polygon points="760.0,825.0 756.5,818.0 763.5,818.0" fill="#3B82F6"/>
<line x1="760" y1="855" x2="760" y2="885" stroke="#3B82F6" stroke-width="3"/>
<line x1="760" y1="885" x2="760" y2="940" stroke="#3B82F6" stroke-width="3" stroke-dasharray="3 6"/>
<line x1="760" y1="940" x2="760" y2="945" stroke="#3B82F6" stroke-width="3"/>
<polygon points="760.0,950.0 756.5,943.0 763.5,943.0" fill="#3B82F6"/>
<rect x="90" y="50" width="80" height="44" rx="6" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.45"/>
<text x="130" y="70" font-size="15" text-anchor="middle" font-weight="600" fill="currentColor">I</text>
<text x="130" y="87" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.6">ID 40</text>
<line x1="130" y1="94" x2="130" y2="113" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="130.0,118.0 126.5,111.0 133.5,111.0" fill="currentColor"/>
<line x1="130" y1="146" x2="130" y2="165" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="130.0,170.0 126.5,163.0 133.5,163.0" fill="currentColor"/>
<rect x="94" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="106" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.27999999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="118" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.35" stroke="#3B82F6" stroke-width="1"/>
<rect x="130" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.24499999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="142" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.315" stroke="#3B82F6" stroke-width="1"/>
<rect x="154" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="230" y="50" width="80" height="44" rx="6" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.45"/>
<text x="270" y="70" font-size="15" text-anchor="middle" font-weight="600" fill="currentColor">ran</text>
<text x="270" y="87" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.6">ID 4966</text>
<line x1="270" y1="94" x2="270" y2="113" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="270.0,118.0 266.5,111.0 273.5,111.0" fill="currentColor"/>
<line x1="270" y1="146" x2="270" y2="165" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="270.0,170.0 266.5,163.0 273.5,163.0" fill="currentColor"/>
<rect x="234" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="246" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.27999999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="258" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.35" stroke="#3B82F6" stroke-width="1"/>
<rect x="270" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.24499999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="282" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.315" stroke="#3B82F6" stroke-width="1"/>
<rect x="294" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="370" y="50" width="80" height="44" rx="6" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.45"/>
<text x="410" y="70" font-size="15" text-anchor="middle" font-weight="600" fill="currentColor">up</text>
<text x="410" y="87" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.6">ID 510</text>
<line x1="410" y1="94" x2="410" y2="113" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="410.0,118.0 406.5,111.0 413.5,111.0" fill="currentColor"/>
<line x1="410" y1="146" x2="410" y2="165" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="410.0,170.0 406.5,163.0 413.5,163.0" fill="currentColor"/>
<rect x="374" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="386" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.27999999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="398" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.35" stroke="#3B82F6" stroke-width="1"/>
<rect x="410" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.24499999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="422" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.315" stroke="#3B82F6" stroke-width="1"/>
<rect x="434" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="720" y="50" width="80" height="44" rx="6" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.45"/>
<text x="760" y="70" font-size="15" text-anchor="middle" font-weight="600" fill="currentColor">the</text>
<text x="760" y="87" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.6">ID 262</text>
<line x1="760" y1="94" x2="760" y2="113" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="760.0,118.0 756.5,111.0 763.5,111.0" fill="currentColor"/>
<line x1="760" y1="146" x2="760" y2="165" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<polygon points="760.0,170.0 756.5,163.0 763.5,163.0" fill="currentColor"/>
<rect x="724" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="736" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.27999999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="748" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.35" stroke="#3B82F6" stroke-width="1"/>
<rect x="760" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.24499999999999997" stroke="#3B82F6" stroke-width="1"/>
<rect x="772" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.315" stroke="#3B82F6" stroke-width="1"/>
<rect x="784" y="170" width="12" height="14" fill="#3B82F6" fill-opacity="0.21" stroke="#3B82F6" stroke-width="1"/>
<rect x="70" y="118" width="750" height="28" rx="6" fill="currentColor" fill-opacity="0.06" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.4"/>
<text x="585" y="137" font-size="12" text-anchor="middle" font-weight="500" fill="currentColor">Embedding matrix (shared by all positions)</text>
<text x="1080" y="68" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Token IDs</text>
<text x="1080" y="120" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">One shared lookup table</text>
<text x="1080" y="136" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">for every position</text>
<text x="1080" y="162" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Input embeddings:</text>
<text x="1080" y="178" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">each token starts its</text>
<text x="1080" y="194" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">own residual stream</text>
<rect x="50" y="210" width="1010" height="320" rx="14" fill="currentColor" fill-opacity="0.025" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.5"/>
<line x1="130" y1="295" x2="144" y2="295" stroke="#3B82F6" stroke-width="2"/>
<rect x="144" y="280" width="60" height="30" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2" stroke-opacity="1"/>
<text x="174" y="299" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">Attn</text>
<path d="M174,310 V360 H139" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="139.0,360.0 146.0,356.5 146.0,363.5" fill="#10B981"/>
<circle cx="130" cy="360" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="125" y1="360" x2="135" y2="360" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="130" y1="355" x2="130" y2="365" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="130" y1="425" x2="144" y2="425" stroke="#3B82F6" stroke-width="2"/>
<rect x="144" y="410" width="60" height="30" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.2" stroke-opacity="1"/>
<text x="174" y="429" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">FFN</text>
<path d="M174,440 V490 H139" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="139.0,490.0 146.0,486.5 146.0,493.5" fill="#10B981"/>
<circle cx="130" cy="490" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="125" y1="490" x2="135" y2="490" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="130" y1="485" x2="130" y2="495" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="295" x2="284" y2="295" stroke="#3B82F6" stroke-width="2"/>
<rect x="284" y="280" width="60" height="30" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2" stroke-opacity="1"/>
<text x="314" y="299" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">Attn</text>
<path d="M314,310 V360 H279" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="279.0,360.0 286.0,356.5 286.0,363.5" fill="#10B981"/>
<circle cx="270" cy="360" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="265" y1="360" x2="275" y2="360" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="355" x2="270" y2="365" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="425" x2="284" y2="425" stroke="#3B82F6" stroke-width="2"/>
<rect x="284" y="410" width="60" height="30" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.2" stroke-opacity="1"/>
<text x="314" y="429" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">FFN</text>
<path d="M314,440 V490 H279" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="279.0,490.0 286.0,486.5 286.0,493.5" fill="#10B981"/>
<circle cx="270" cy="490" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="265" y1="490" x2="275" y2="490" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="485" x2="270" y2="495" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="295" x2="424" y2="295" stroke="#3B82F6" stroke-width="2"/>
<rect x="424" y="280" width="60" height="30" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2" stroke-opacity="1"/>
<text x="454" y="299" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">Attn</text>
<path d="M454,310 V360 H419" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="419.0,360.0 426.0,356.5 426.0,363.5" fill="#10B981"/>
<circle cx="410" cy="360" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="405" y1="360" x2="415" y2="360" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="355" x2="410" y2="365" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="425" x2="424" y2="425" stroke="#3B82F6" stroke-width="2"/>
<rect x="424" y="410" width="60" height="30" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.2" stroke-opacity="1"/>
<text x="454" y="429" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">FFN</text>
<path d="M454,440 V490 H419" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="419.0,490.0 426.0,486.5 426.0,493.5" fill="#10B981"/>
<circle cx="410" cy="490" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="405" y1="490" x2="415" y2="490" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="485" x2="410" y2="495" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<path d="M130,226 H596 Q610,226 610,240 V270" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="610.0,270.0 606.5,263.0 613.5,263.0" fill="#E8833A"/>
<circle cx="130" cy="226" r="3.5" fill="#E8833A"/>
<path d="M270,238 H561 Q575,238 575,252 V270" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="575.0,270.0 571.5,263.0 578.5,263.0" fill="#E8833A"/>
<circle cx="270" cy="238" r="3.5" fill="#E8833A"/>
<path d="M410,250 H526 Q540,250 540,264 V270" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="540.0,270.0 536.5,263.0 543.5,263.0" fill="#E8833A"/>
<circle cx="410" cy="250" r="3.5" fill="#E8833A"/>
<path d="M760,238 H659 Q645,238 645,252 V270" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="645.0,270.0 641.5,263.0 648.5,263.0" fill="#E8833A"/>
<circle cx="760" cy="238" r="3.5" fill="#E8833A"/>
<rect x="505" y="270" width="180" height="50" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.5" stroke-opacity="1"/>
<text x="595" y="292" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">Attention</text>
<text x="595" y="309" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.7">causal: itself + earlier</text>
<path d="M595,320 V360 H700" fill="none" stroke="#10B981" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="700.0,360.0 693.0,363.5 693.0,356.5" fill="#10B981"/>
<text x="600" y="342" font-size="11" text-anchor="start" font-weight="normal" fill="#10B981" fill-opacity="0.75">update</text>
<text x="768" y="308" font-size="11" text-anchor="start" font-weight="normal" fill="#3B82F6" fill-opacity="0.75">residual</text>
<rect x="700" y="345" width="120" height="30" rx="6" fill="currentColor" fill-opacity="0.08" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.55"/>
<text x="760" y="365" font-size="12" text-anchor="middle" font-weight="500" fill="currentColor">Add &amp; Norm</text>
<path d="M760,387 H950 V400" fill="none" stroke="#3B82F6" stroke-width="2" stroke-linejoin="round"/>
<polygon points="950.0,400.0 946.5,393.0 953.5,393.0" fill="#3B82F6"/>
<circle cx="760" cy="387" r="3.5" fill="#3B82F6"/>
<rect x="870" y="400" width="160" height="50" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.5" stroke-opacity="1"/>
<text x="950" y="422" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">FFN</text>
<text x="950" y="439" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.7">this position only</text>
<path d="M950,450 V490 H820" fill="none" stroke="#10B981" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="820.0,490.0 827.0,486.5 827.0,493.5" fill="#10B981"/>
<text x="890" y="482" font-size="11" text-anchor="start" font-weight="normal" fill="#10B981" fill-opacity="0.75">update</text>
<text x="768" y="438" font-size="11" text-anchor="start" font-weight="normal" fill="#3B82F6" fill-opacity="0.75">residual</text>
<rect x="700" y="475" width="120" height="30" rx="6" fill="currentColor" fill-opacity="0.08" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.55"/>
<text x="760" y="495" font-size="12" text-anchor="middle" font-weight="500" fill="currentColor">Add &amp; Norm</text>
<text x="1080" y="230" font-size="14" text-anchor="start" font-weight="600" fill="currentColor">Transformer block 1</text>
<text x="1080" y="252" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Positions 1–3: compact view</text>
<text x="1080" y="268" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Position 4 (“the”): expanded</text>
<text x="1080" y="290" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Attention mixes across</text>
<text x="1080" y="306" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">positions: “the” reads from</text>
<text x="1080" y="322" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">I, ran, up and itself —</text>
<text x="1080" y="338" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">never from future tokens</text>
<text x="1080" y="360" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Updates are added to the</text>
<text x="1080" y="376" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">stream, not replacing it</text>
<text x="1080" y="415" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">FFN works on each</text>
<text x="1080" y="431" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">position independently —</text>
<text x="1080" y="447" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">no cross-position input</text>
<rect x="50" y="560" width="1010" height="320" rx="14" fill="currentColor" fill-opacity="0.025" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.5"/>
<line x1="130" y1="645" x2="144" y2="645" stroke="#3B82F6" stroke-width="2"/>
<rect x="144" y="630" width="60" height="30" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2" stroke-opacity="1"/>
<text x="174" y="649" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">Attn</text>
<path d="M174,660 V710 H139" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="139.0,710.0 146.0,706.5 146.0,713.5" fill="#10B981"/>
<circle cx="130" cy="710" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="125" y1="710" x2="135" y2="710" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="130" y1="705" x2="130" y2="715" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="130" y1="775" x2="144" y2="775" stroke="#3B82F6" stroke-width="2"/>
<rect x="144" y="760" width="60" height="30" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.2" stroke-opacity="1"/>
<text x="174" y="779" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">FFN</text>
<path d="M174,790 V840 H139" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="139.0,840.0 146.0,836.5 146.0,843.5" fill="#10B981"/>
<circle cx="130" cy="840" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="125" y1="840" x2="135" y2="840" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="130" y1="835" x2="130" y2="845" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="645" x2="284" y2="645" stroke="#3B82F6" stroke-width="2"/>
<rect x="284" y="630" width="60" height="30" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2" stroke-opacity="1"/>
<text x="314" y="649" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">Attn</text>
<path d="M314,660 V710 H279" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="279.0,710.0 286.0,706.5 286.0,713.5" fill="#10B981"/>
<circle cx="270" cy="710" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="265" y1="710" x2="275" y2="710" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="705" x2="270" y2="715" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="775" x2="284" y2="775" stroke="#3B82F6" stroke-width="2"/>
<rect x="284" y="760" width="60" height="30" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.2" stroke-opacity="1"/>
<text x="314" y="779" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">FFN</text>
<path d="M314,790 V840 H279" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="279.0,840.0 286.0,836.5 286.0,843.5" fill="#10B981"/>
<circle cx="270" cy="840" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="265" y1="840" x2="275" y2="840" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="270" y1="835" x2="270" y2="845" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="645" x2="424" y2="645" stroke="#3B82F6" stroke-width="2"/>
<rect x="424" y="630" width="60" height="30" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2" stroke-opacity="1"/>
<text x="454" y="649" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">Attn</text>
<path d="M454,660 V710 H419" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="419.0,710.0 426.0,706.5 426.0,713.5" fill="#10B981"/>
<circle cx="410" cy="710" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="405" y1="710" x2="415" y2="710" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="705" x2="410" y2="715" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="775" x2="424" y2="775" stroke="#3B82F6" stroke-width="2"/>
<rect x="424" y="760" width="60" height="30" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.2" stroke-opacity="1"/>
<text x="454" y="779" font-size="11" text-anchor="middle" font-weight="500" fill="currentColor">FFN</text>
<path d="M454,790 V840 H419" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="419.0,840.0 426.0,836.5 426.0,843.5" fill="#10B981"/>
<circle cx="410" cy="840" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="405" y1="840" x2="415" y2="840" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="410" y1="835" x2="410" y2="845" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<path d="M130,576 H596 Q610,576 610,590 V620" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="610.0,620.0 606.5,613.0 613.5,613.0" fill="#E8833A"/>
<circle cx="130" cy="576" r="3.5" fill="#E8833A"/>
<path d="M270,588 H561 Q575,588 575,602 V620" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="575.0,620.0 571.5,613.0 578.5,613.0" fill="#E8833A"/>
<circle cx="270" cy="588" r="3.5" fill="#E8833A"/>
<path d="M410,600 H526 Q540,600 540,614 V620" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="540.0,620.0 536.5,613.0 543.5,613.0" fill="#E8833A"/>
<circle cx="410" cy="600" r="3.5" fill="#E8833A"/>
<path d="M760,588 H659 Q645,588 645,602 V620" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-linejoin="round"/>
<polygon points="645.0,620.0 641.5,613.0 648.5,613.0" fill="#E8833A"/>
<circle cx="760" cy="588" r="3.5" fill="#E8833A"/>
<rect x="505" y="620" width="180" height="50" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.5" stroke-opacity="1"/>
<text x="595" y="642" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">Attention</text>
<text x="595" y="659" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.7">causal: itself + earlier</text>
<path d="M595,670 V710 H700" fill="none" stroke="#10B981" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="700.0,710.0 693.0,713.5 693.0,706.5" fill="#10B981"/>
<text x="600" y="692" font-size="11" text-anchor="start" font-weight="normal" fill="#10B981" fill-opacity="0.75">update</text>
<text x="768" y="658" font-size="11" text-anchor="start" font-weight="normal" fill="#3B82F6" fill-opacity="0.75">residual</text>
<rect x="700" y="695" width="120" height="30" rx="6" fill="currentColor" fill-opacity="0.08" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.55"/>
<text x="760" y="715" font-size="12" text-anchor="middle" font-weight="500" fill="currentColor">Add &amp; Norm</text>
<path d="M760,737 H950 V750" fill="none" stroke="#3B82F6" stroke-width="2" stroke-linejoin="round"/>
<polygon points="950.0,750.0 946.5,743.0 953.5,743.0" fill="#3B82F6"/>
<circle cx="760" cy="737" r="3.5" fill="#3B82F6"/>
<rect x="870" y="750" width="160" height="50" rx="6" fill="#8B5CF6" fill-opacity="0.16" stroke="#8B5CF6" stroke-width="1.5" stroke-opacity="1"/>
<text x="950" y="772" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">FFN</text>
<text x="950" y="789" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.7">this position only</text>
<path d="M950,800 V840 H820" fill="none" stroke="#10B981" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="820.0,840.0 827.0,836.5 827.0,843.5" fill="#10B981"/>
<text x="890" y="832" font-size="11" text-anchor="start" font-weight="normal" fill="#10B981" fill-opacity="0.75">update</text>
<text x="768" y="788" font-size="11" text-anchor="start" font-weight="normal" fill="#3B82F6" fill-opacity="0.75">residual</text>
<rect x="700" y="825" width="120" height="30" rx="6" fill="currentColor" fill-opacity="0.08" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.55"/>
<text x="760" y="845" font-size="12" text-anchor="middle" font-weight="500" fill="currentColor">Add &amp; Norm</text>
<text x="1080" y="580" font-size="14" text-anchor="start" font-weight="600" fill="currentColor">Transformer block 2</text>
<text x="1080" y="602" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Same structure,</text>
<text x="1080" y="618" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">its own weights</text>
<text x="585" y="912" font-size="22" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.7">⋮</text>
<text x="585" y="932" font-size="12" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.75" font-style="italic">repeat through block L</text>
<text x="1080" y="918" font-size="14" text-anchor="start" font-weight="600" fill="currentColor">… blocks 3 to L</text>
<rect x="94" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.27" stroke="#3B82F6" stroke-width="1"/>
<rect x="106" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.36000000000000004" stroke="#3B82F6" stroke-width="1"/>
<rect x="118" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.45" stroke="#3B82F6" stroke-width="1"/>
<rect x="130" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.315" stroke="#3B82F6" stroke-width="1"/>
<rect x="142" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.405" stroke="#3B82F6" stroke-width="1"/>
<rect x="154" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.27" stroke="#3B82F6" stroke-width="1"/>
<line x1="130" y1="964" x2="130" y2="983" stroke="currentColor" stroke-width="1.5" stroke-dasharray="3 3" stroke-opacity="0.35"/>
<polygon points="130.0,988.0 126.5,981.0 133.5,981.0" fill="currentColor"/>
<text x="130" y="1004" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.4" font-style="italic">logits</text>
<rect x="234" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.27" stroke="#3B82F6" stroke-width="1"/>
<rect x="246" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.36000000000000004" stroke="#3B82F6" stroke-width="1"/>
<rect x="258" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.45" stroke="#3B82F6" stroke-width="1"/>
<rect x="270" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.315" stroke="#3B82F6" stroke-width="1"/>
<rect x="282" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.405" stroke="#3B82F6" stroke-width="1"/>
<rect x="294" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.27" stroke="#3B82F6" stroke-width="1"/>
<line x1="270" y1="964" x2="270" y2="983" stroke="currentColor" stroke-width="1.5" stroke-dasharray="3 3" stroke-opacity="0.35"/>
<polygon points="270.0,988.0 266.5,981.0 273.5,981.0" fill="currentColor"/>
<text x="270" y="1004" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.4" font-style="italic">logits</text>
<rect x="374" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.27" stroke="#3B82F6" stroke-width="1"/>
<rect x="386" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.36000000000000004" stroke="#3B82F6" stroke-width="1"/>
<rect x="398" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.45" stroke="#3B82F6" stroke-width="1"/>
<rect x="410" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.315" stroke="#3B82F6" stroke-width="1"/>
<rect x="422" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.405" stroke="#3B82F6" stroke-width="1"/>
<rect x="434" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.27" stroke="#3B82F6" stroke-width="1"/>
<line x1="410" y1="964" x2="410" y2="983" stroke="currentColor" stroke-width="1.5" stroke-dasharray="3 3" stroke-opacity="0.35"/>
<polygon points="410.0,988.0 406.5,981.0 413.5,981.0" fill="currentColor"/>
<text x="410" y="1004" font-size="11" text-anchor="middle" font-weight="normal" fill="currentColor" fill-opacity="0.4" font-style="italic">logits</text>
<rect x="724" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.33" stroke="#3B82F6" stroke-width="1"/>
<rect x="736" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.44000000000000006" stroke="#3B82F6" stroke-width="1"/>
<rect x="748" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.55" stroke="#3B82F6" stroke-width="1"/>
<rect x="760" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.385" stroke="#3B82F6" stroke-width="1"/>
<rect x="772" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.49500000000000005" stroke="#3B82F6" stroke-width="1"/>
<rect x="784" y="950" width="12" height="14" fill="#3B82F6" fill-opacity="0.33" stroke="#3B82F6" stroke-width="1"/>
<rect x="716" y="943" width="88" height="28" rx="5" fill="#E8833A" fill-opacity="0" stroke="#E8833A" stroke-width="2" stroke-opacity="1"/>
<text x="1080" y="955" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Final contextual</text>
<text x="1080" y="971" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">embeddings</text>
<line x1="760" y1="971" x2="760" y2="985" stroke="currentColor" stroke-width="1.8" stroke-opacity="0.75"/>
<polygon points="760.0,990.0 756.5,983.0 763.5,983.0" fill="currentColor"/>
<rect x="650" y="990" width="220" height="30" rx="6" fill="currentColor" fill-opacity="0.08" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.55"/>
<text x="760" y="1010" font-size="12" text-anchor="middle" font-weight="500" fill="currentColor">Output projection (unembedding)</text>
<line x1="760" y1="1020" x2="760" y2="1040" stroke="currentColor" stroke-width="1.8" stroke-opacity="0.75"/>
<polygon points="760.0,1045.0 756.5,1038.0 763.5,1038.0" fill="currentColor"/>
<rect x="640" y="1045" width="260" height="108" rx="6" fill="#E8833A" fill-opacity="0.07" stroke="#E8833A" stroke-width="1.5" stroke-opacity="0.6"/>
<text x="660" y="1065" font-size="12" text-anchor="start" font-weight="600" fill="currentColor">Next-token logits</text>
<text x="660" y="1086" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor">stairs</text>
<rect x="725" y="1076" width="116.6" height="11" rx="2" fill="#E8833A" fill-opacity="0.9"/>
<text x="849.6" y="1086" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.8">5.3</text>
<text x="660" y="1106" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor">hill</text>
<rect x="725" y="1096" width="112.2" height="11" rx="2" fill="#E8833A" fill-opacity="0.7"/>
<text x="845.2" y="1106" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.8">5.1</text>
<text x="660" y="1126" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor">street</text>
<rect x="725" y="1116" width="99.0" height="11" rx="2" fill="#E8833A" fill-opacity="0.5"/>
<text x="832" y="1126" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.8">4.5</text>
<text x="660" y="1144" font-size="13" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.7">…</text>
<text x="1080" y="1010" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">Only the last position is</text>
<text x="1080" y="1026" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">needed to predict the</text>
<text x="1080" y="1042" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor" fill-opacity="0.75">next token</text>
<line x1="60" y1="1046" x2="94" y2="1046" stroke="#3B82F6" stroke-width="3"/>
<text x="104" y="1050" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor">Residual stream (one per token, flows down)</text>
<line x1="60" y1="1068" x2="94" y2="1068" stroke="#E8833A" stroke-width="1.8"/>
<text x="104" y="1072" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor">Attention: information flows across positions</text>
<line x1="60" y1="1090" x2="94" y2="1090" stroke="#10B981" stroke-width="2.2"/>
<text x="104" y="1094" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor">Update added back to the residual stream</text>
<circle cx="77" cy="1112" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="72" y1="1112" x2="82" y2="1112" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="77" y1="1107" x2="77" y2="1117" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<text x="104" y="1116" font-size="12" text-anchor="start" font-weight="normal" fill="currentColor">Add &amp; Norm (compact view)</text>
</svg>


**The key idea:** The residual stream is never discarded. Each layer reads the current representation, computes a contribution, and adds it back. Over many blocks, the embeddings gradually become contextual representations useful for predicting the next token.

Let's inspect the residual stream after each block to see this process in action.

In [ ]:
CONTEXT = "The transformer"
token_ids = encode(CONTEXT)[np.newaxis, :] # newaxis is for batch dimension. Required even though we have just one token sequence

# --- Map token IDs to embeddings ---

x = tu.embedding(token_ids, params["embedding"])

# --- Positional Encoding (we can mostly ignore for now) ---
positions = tu.positional_encoding(
    seq_len=token_ids.shape[-1],
    d_model=x.shape[-1],
    dtype=x.dtype,
)
x = x + positions

# --- Pass the embedding through the transformer blocks ---

block_inputs = []
block_caches = []
for block_params in params["blocks"]:
    block_inputs.append(x)
    x, block_cache = tu.transformer_block(x, block_params)
    block_caches.append(block_cache)

before_final_norm = x
x, final_norm_cache = tu.rmsnorm(x, params["final_norm"])
embeddings = x

# --- Project embeddings to logits ---

def see_logits(label, x, output_weight, pos=-1):
    logits = tu.matmul(x, output_weight)

    print(label)
    top_10 = np.argsort(logits[0, pos])[::-1][:10]
    for i in top_10:
        print(f"\"{decode([i])}\" - (id = {i}): {logits[0, pos, i]:.3f}")
    print("_"*50)

# --- We can actually project the intermediate residual states into logits. Let's trace it down the network for a single position ---

pos = -1
see_logits("Embedding + positional encoding", block_inputs[0], params["output_weight"], pos=pos)
see_logits("After block 0", block_inputs[1], params["output_weight"], pos=pos)
see_logits("After block 1", block_inputs[2], params["output_weight"], pos=pos)
see_logits("After block 2 (before final norm)", before_final_norm, params["output_weight"], pos=pos)
see_logits("After final norm (what the model actually uses)", embeddings, params["output_weight"], pos=pos)

**Nearest neighbors in embedding space**

Each word's embedding is just a vector in $\mathbb R^{d_\text{model}}$ — a direction and a magnitude, nothing more. If meaning really is encoded as *direction*, then words with related meaning should point similar ways, and cosine similarity between embedding rows should surface them.

But what does it mean for [direction to encode meaning?](https://www.youtube.com/shorts/FJtFZwbvkI4)

---

**Cosine similarity, briefly**

For two vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^d$, the dot product is equal to the sum of the pointwise product.  
For example, if your vectors are $(a, b)$ and $(c,d)$, then the dot product is just a single scalar (number): $ac + cd$.

It turns out this can also be expressed using the *cosine of the angle between the vectors:*

$$
\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\,\cos\theta
$$

where $\theta$ is the angle between them. If we normalize both vectors to unit length first ($\hat{\mathbf{a}} = \mathbf{a}/\|\mathbf{a}\|$,
$\hat{\mathbf{b}} = \mathbf{b}/\|\mathbf{b}\|$), the magnitude terms disappear and we're left with exactly the angle:

$$
\hat{\mathbf{a}} \cdot \hat{\mathbf{b}} = \cos\theta
$$

This is **cosine similarity** - the dot product of the normalized vectors, which is just $\cos\theta$. It only cares about *direction*, not
length, which is exactly what we want if meaning lives in direction rather than magnitude.

Since $\cos\theta$ ranges over $[-1, 1]$:

- $\theta = 0°$ (same direction) $\Rightarrow \cos\theta = 1$ - the vectors point the same way.
- $\theta = 90°$ (orthogonal) $\Rightarrow \cos\theta = 0$ - the vectors share no direction in common.
- $\theta = 180°$ (opposite direction) $\Rightarrow \cos\theta = -1$ - the vectors point exactly opposite ways.

So the smaller the angle between two vectors, the larger $\cos\theta$ - and the higher the "similarity" score we assign them.

---

Pick a `QUERY_WORD` below and look at its 10 closest neighbors by cosine similarity (the word itself will rank first — a sanity check that this is in fact what cosine similarity search looks like). 

In [ ]:
QUERY_WORD = "learning"

query_tokens = encode(QUERY_WORD if QUERY_WORD.startswith(" ") else " " + QUERY_WORD) # Use space-leading token since there are more examples of them in the data
assert len(query_tokens) == 1, f"QUERY_WORD must tokenize to a single token, got {len(query_tokens)}"
query_token = query_tokens[0]
print(f"Queried token: \"{decode([query_token])}\" (id = {query_token})")

embedding_matrix = params["embedding"]
unit_embeddings = embedding_matrix / np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
similarities = unit_embeddings @ unit_embeddings[query_token]

top_10 = np.argsort(similarities)[::-1][:10]
for i in top_10:
  print(f"\"{decode([i])}\" - (id = {i}): similarity = {similarities[i]:.3f}")

**Sequence-level semantic search**

A single token's embedding is constant. Once it's learned through training, it doesn't change. But as a sequence passes through the transformer blocks, attention lets each position gather information from every other token, so the residual state at the final position ends up encoding something about the *whole sentence*, not just the last word.

So the same trick from before (cosine similarity between vectors) should work one level up: embed a handful of candidate sentences (`KEYS`) and a `QUERY`, and rank the keys by how close their final residual state is to the query's. Toggle `SUM_ACROSS_LAYERS` to compare using only the last block's output vs. summing the residual contribution from every block.

One hypothesis worth testing with that toggle: the final block's output is most directly tied to predicting the next token, while earlier blocks may carry more general information accumulated from other tokens — so summing across blocks *might* improve the search. See if that holds up when you try it.

(One caveat: positional encoding is absolute here, so a sentence's final-token vector is colored by *how long* the sentence is, not just what it means. This is something to keep in mind when judging the rankings.)

In [ ]:
KEYS = [
    "The transformer processes sequences of tokens.",
    "Attention lets each token look at every other token.",
    "A neural network learns patterns from data.",
    "The model was trained on a large corpus of text.",
    "Gradient descent updates the weights to reduce loss.",
    "Embeddings map discrete tokens into continuous vectors.",
    "The recurrent network struggled with long sequences.",
    "Backpropagation computes gradients through the network.",
    "She walked her dog in the park.",
    "He cooked dinner for his family.",
    "The children played soccer after school.",
    "It rained heavily throughout the weekend.",
    "She planted tomatoes in the garden this spring.",
    "The chef prepared a five-course meal.",
    "They watched a movie at the theater.",
    "He fixed the leaky faucet in the kitchen.",
]
QUERY = "A neural network reads a sequence of words."

SUM_ACROSS_LAYERS = False  # True: sum the residual state after every block instead of just using the final one

def embed_sequence(text):
    token_ids = encode(text)[np.newaxis, :]
    logits, cache = tu.transformer(token_ids, params)

    if SUM_ACROSS_LAYERS:
        layer_states = cache["block_inputs"][1:] + [cache["before_final_norm"]]
        vector = sum(state[0, -1] for state in layer_states)
    else:
        vector = cache["final_normalized"][0, -1]

    return vector / np.linalg.norm(vector)

query_vector = embed_sequence(QUERY)
key_vectors = np.stack([embed_sequence(key) for key in KEYS])

similarities = key_vectors @ query_vector
ranking = np.argsort(similarities)[::-1]

print(f'QUERY: "{QUERY}"\n')
for i in ranking:
    print(f'{similarities[i]:.3f}  "{KEYS[i]}"')

# Why don't we need trillions of dimensions to embed arbitrary combinations of meaning?

How many vectors in $\mathbb{R}^d$ can be pairwise *exactly* orthogonal? Just $d$ - that's the whole point of a basis. By that logic,
$d_\text{model} = 64$ should only be able to represent 64 independent "directions of meaning." That seems far too few for everything words need
to encode.

But embeddings don't need to be *exactly* orthogonal to stay distinguishable - they just need to be close enough to it that one direction 
doesn't look like another. And "almost orthogonal" is a much weaker, much more abundant condition than "exactly orthogonal."

Here's the intuition: pick a random unit vector in $\mathbb{R}^d$, and ask how likely it is to land within some small cosine similarity 
$\varepsilon$ of a fixed direction. As $d$ grows, high-dimensional space becomes increasingly "spread out" - random directions concentrate 
tightly around $90°$ from one another, so the probability of accidentally landing close to any one fixed direction *shrinks exponentially with 
$d$*:

$$
P(|\cos\theta| > \varepsilon) \lesssim e^{-c \, d \, \varepsilon^2}
$$

for some constant $c$. Flip that around with a union bound over all pairs, and you get the number of vectors $N$ you can pack into 
$\mathbb{R}^d$ while keeping every pairwise angle within $\varepsilon$ of orthogonal:

$$
N \lesssim e^{\,c' \, d \, \varepsilon^2}
$$

The count of *nearly*-orthogonal directions grows **exponentially in $d$**, not linearly like the exact-orthogonality count. So instead of 64 
usable directions, a 64-dimensional space can hold an astronomically larger number of "good enough" ones - and by the time $d$ reaches a few 
hundred, that number routinely blows past estimates like the number of atoms in the observable universe ($\sim 10^{80}$).

This is the geometric fact that makes dense embeddings workable at all: meaning doesn't need its own orthogonal axis, just its own *roughly 
separate* direction - and in high dimensions, there's room for far more of those than you'd ever guess from the dimension count alone.